In [2]:
# Sales Data Analysis
# Week 3 Project - Introduction to Data Analysis
# Author: Your Name
# Date: December 25, 2025

"""

This program analyzes sales data to extract meaningful insights.
It demonstrates pandas basics: loading data, exploring, cleaning, and analyzing.
"""

import pandas as pd
import numpy as np

# ============================================================
# SECTION 1: LOAD AND EXPLORE DATA
# ============================================================

def load_data(filename):
    """
    Load the sales data from CSV file
    Parameters: filename (str) - path to CSV file
    Returns: DataFrame with sales data
    """
    try:
        df = pd.read_csv('/content/sales_data.csv')
        print("✅ Data loaded successfully!")
        return df
    except FileNotFoundError:
        print(f"❌ Error: File '{filename}' not found!")
        return None
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None

def display_basic_info(df):
    """
    Display basic information about the dataset
    Parameters: df (DataFrame) - sales data
    """
    print("\n" + "=" * 60)
    print("📊 BASIC DATASET INFORMATION")
    print("=" * 60)

    # Shape of dataset
    print(f"\n📏 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")

    # Column names and types
    print("\n📋 Column Information:")
    print(df.dtypes)

    # First few rows
    print("\n👀 First 5 Rows of Data:")
    print(df.head())

    # Basic statistics
    print("\n📈 Statistical Summary:")
    print(df.describe())

    # Missing values
    print("\n🔍 Missing Values Check:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("✅ No missing values found!")

# ============================================================
# SECTION 2: DATA CLEANING
# ============================================================

def clean_data(df):
    """
    Clean the data by handling missing values and duplicates
    Parameters: df (DataFrame) - sales data
    Returns: cleaned DataFrame
    """
    print("\n" + "=" * 60)
    print("🧹 DATA CLEANING")
    print("=" * 60)

    # Make a copy to avoid modifying original
    df_clean = df.copy()

    # Check for duplicates
    duplicates = df_clean.duplicated().sum()
    print(f"\n🔄 Duplicate rows found: {duplicates}")
    if duplicates > 0:
        df_clean = df_clean.drop_duplicates()
        print(f"✅ Removed {duplicates} duplicate rows")

    # Check for missing values
    missing_before = df_clean.isnull().sum().sum()
    print(f"\n❓ Missing values before cleaning: {missing_before}")

    # Handle missing values in numeric columns
    numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        if df_clean[col].isnull().sum() > 0:
            # Fill with median for numeric columns
            median_value = df_clean[col].median()
            df_clean[col].fillna(median_value, inplace=True)
            print(f"  ✅ Filled missing values in '{col}' with median: {median_value}")

    # Handle missing values in text columns
    text_columns = df_clean.select_dtypes(include=['object']).columns
    for col in text_columns:
        if df_clean[col].isnull().sum() > 0:
            # Fill with mode (most frequent value) for text columns
            mode_value = df_clean[col].mode()[0] if not df_clean[col].mode().empty else "Unknown"
            df_clean[col].fillna(mode_value, inplace=True)
            print(f"  ✅ Filled missing values in '{col}' with mode: {mode_value}")

    missing_after = df_clean.isnull().sum().sum()
    print(f"\n❓ Missing values after cleaning: {missing_after}")
    print(f"✅ Data cleaning completed! Shape: {df_clean.shape}")

    return df_clean

# ============================================================
# SECTION 3: SALES ANALYSIS
# ============================================================

def analyze_total_sales(df):
    """
    Calculate total sales revenue
    Parameters: df (DataFrame) - sales data
    Returns: float - total revenue
    """
    total_revenue = df['Total_Sales'].sum()
    print(f"\n💰 Total Revenue: ₹{total_revenue:,.2f}")
    return total_revenue

def analyze_product_performance(df):
    """
    Analyze which products perform best
    Parameters: df (DataFrame) - sales data
    Returns: DataFrame - product sales summary
    """
    print("\n" + "=" * 60)
    print("📦 PRODUCT PERFORMANCE ANALYSIS")
    print("=" * 60)

    # Group by product and calculate metrics
    product_sales = df.groupby('Product').agg({
        'Quantity': 'sum',
        'Total_Sales': 'sum',
        'Customer_ID': 'count'
    }).reset_index()

    # Rename columns for clarity
    product_sales.columns = ['Product', 'Total_Quantity_Sold', 'Total_Revenue', 'Number_of_Orders']

    # Sort by revenue
    product_sales = product_sales.sort_values('Total_Revenue', ascending=False)

    # Display results
    print("\n📊 Product Sales Summary:")
    print(product_sales.to_string(index=False))

    # Best selling product by revenue
    best_product = product_sales.iloc[0]
    print(f"\n🏆 Best Selling Product (by Revenue):")
    print(f"   Product: {best_product['Product']}")
    print(f"   Revenue: ₹{best_product['Total_Revenue']:,.2f}")
    print(f"   Quantity Sold: {best_product['Total_Quantity_Sold']}")

    return product_sales

def analyze_regional_performance(df):
    """
    Analyze sales performance by region
    Parameters: df (DataFrame) - sales data
    Returns: DataFrame - regional sales summary
    """
    print("\n" + "=" * 60)
    print("🌍 REGIONAL PERFORMANCE ANALYSIS")
    print("=" * 60)

    # Group by region
    regional_sales = df.groupby('Region').agg({
        'Total_Sales': ['sum', 'mean', 'count']
    }).reset_index()

    # Flatten column names
    regional_sales.columns = ['Region', 'Total_Revenue', 'Average_Sale', 'Number_of_Sales']

    # Sort by revenue
    regional_sales = regional_sales.sort_values('Total_Revenue', ascending=False)

    print("\n📍 Regional Sales Summary:")
    for idx, row in regional_sales.iterrows():
        print(f"\n{row['Region']}:")
        print(f"  💰 Total Revenue: ₹{row['Total_Revenue']:,.2f}")
        print(f"  📊 Average Sale: ₹{row['Average_Sale']:,.2f}")
        print(f"  📈 Number of Sales: {int(row['Number_of_Sales'])}")

    # Best performing region
    best_region = regional_sales.iloc[0]
    print(f"\n🏆 Top Performing Region: {best_region['Region']}")
    print(f"   Revenue: ₹{best_region['Total_Revenue']:,.2f}")

    return regional_sales

def analyze_customer_insights(df):
    """
    Analyze customer purchasing patterns
    Parameters: df (DataFrame) - sales data
    Returns: dict - customer insights
    """
    print("\n" + "=" * 60)
    print("👥 CUSTOMER INSIGHTS")
    print("=" * 60)

    # Unique customers
    unique_customers = df['Customer_ID'].nunique()
    print(f"\n👤 Total Unique Customers: {unique_customers}")

    # Average order value
    avg_order_value = df['Total_Sales'].mean()
    print(f"💳 Average Order Value: ₹{avg_order_value:,.2f}")

    # Customer spending analysis
    customer_spending = df.groupby('Customer_ID').agg({
        'Total_Sales': 'sum',
        'Quantity': 'sum'
    }).reset_index()

    customer_spending.columns = ['Customer_ID', 'Total_Spent', 'Total_Items']
    customer_spending = customer_spending.sort_values('Total_Spent', ascending=False)

    # Top 5 customers
    print("\n🌟 Top 5 Customers by Spending:")
    for idx, row in customer_spending.head(5).iterrows():
        print(f"   {row['Customer_ID']}: ₹{row['Total_Spent']:,.2f} ({int(row['Total_Items'])} items)")

    return {
        'unique_customers': unique_customers,
        'avg_order_value': avg_order_value,
        'top_customer': customer_spending.iloc[0]['Customer_ID']
    }

def analyze_price_quantity_relationship(df):
    """
    Analyze relationship between price and quantity
    Parameters: df (DataFrame) - sales data
    """
    print("\n" + "=" * 60)
    print("💵 PRICE & QUANTITY ANALYSIS")
    print("=" * 60)

    # Average price
    avg_price = df['Price'].mean()
    max_price = df['Price'].max()
    min_price = df['Price'].min()

    print(f"\n📊 Price Statistics:")
    print(f"   Average Price: ₹{avg_price:,.2f}")
    print(f"   Highest Price: ₹{max_price:,.2f}")
    print(f"   Lowest Price: ₹{min_price:,.2f}")

    # Average quantity per order
    avg_quantity = df['Quantity'].mean()
    total_items_sold = df['Quantity'].sum()

    print(f"\n📦 Quantity Statistics:")
    print(f"   Average Quantity per Order: {avg_quantity:.2f}")
    print(f"   Total Items Sold: {int(total_items_sold)}")

# ============================================================
# SECTION 4: GENERATE COMPREHENSIVE REPORT
# ============================================================

def generate_report(df, filename='analysis_report.md'):
    """
    Generate a comprehensive markdown report
    Parameters:
        df (DataFrame) - sales data
        filename (str) - output filename
    """
    print("\n" + "=" * 60)
    print("📝 GENERATING COMPREHENSIVE REPORT")
    print("=" * 60)

    # Perform all analyses
    total_revenue = analyze_total_sales(df)
    product_data = analyze_product_performance(df)
    regional_data = analyze_regional_performance(df)
    customer_insights = analyze_customer_insights(df)
    analyze_price_quantity_relationship(df)

    # Create report content
    report = f"""# Sales Data Analysis Report
Generated on: December 25, 2025

---

## Executive Summary

This report analyzes {len(df)} sales transactions to provide insights into product performance, regional trends, and customer behavior.

### Key Metrics at a Glance
- **Total Revenue**: ₹{total_revenue:,.2f}
- **Total Transactions**: {len(df)}
- **Unique Customers**: {customer_insights['unique_customers']}
- **Average Order Value**: ₹{customer_insights['avg_order_value']:,.2f}

---

## 1. Product Performance

### Top Products by Revenue
{product_data.to_markdown(index=False)}

### Key Findings
- **Best Selling Product**: {product_data.iloc[0]['Product']}
- **Revenue Generated**: ₹{product_data.iloc[0]['Total_Revenue']:,.2f}
- **Units Sold**: {int(product_data.iloc[0]['Total_Quantity_Sold'])}

---

## 2. Regional Performance

### Sales by Region
{regional_data.to_markdown(index=False)}

### Key Findings
- **Top Performing Region**: {regional_data.iloc[0]['Region']}
- **Regional Revenue**: ₹{regional_data.iloc[0]['Total_Revenue']:,.2f}

---

## 3. Customer Insights

- **Total Unique Customers**: {customer_insights['unique_customers']}
- **Average Order Value**: ₹{customer_insights['avg_order_value']:,.2f}
- **Top Customer**: {customer_insights['top_customer']}

---

## 4. Recommendations

Based on the analysis:

1. **Product Strategy**: Focus on promoting the best-selling products
2. **Regional Strategy**: Allocate more resources to top-performing regions
3. **Customer Retention**: Implement loyalty programs for top customers
4. **Pricing Optimization**: Review pricing strategy for low-performing products

---

## Conclusion

The analysis reveals strong sales performance with clear leaders in product categories and regions. Strategic focus on these areas can drive further growth.

---

*Report generated automatically using Python and Pandas*
"""

    # Save report
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(report)

    print(f"\n✅ Report saved to '{filename}'")
    print("=" * 60)

# ============================================================
# MAIN PROGRAM
# ============================================================

def main():
    """Main function to run the complete analysis"""

    print("\n" + "=" * 60)
    print("🎯 SALES DATA ANALYSIS PROGRAM")
    print("=" * 60)

    # Step 1: Load data
    print("\n[Step 1/5] Loading data...")
    df = load_data('sales_data.csv')

    if df is None:
        print("❌ Cannot proceed without data. Exiting.")
        return

    # Step 2: Display basic info
    print("\n[Step 2/5] Exploring data...")
    display_basic_info(df)

    # Step 3: Clean data
    print("\n[Step 3/5] Cleaning data...")
    df_clean = clean_data(df)

    # Step 4: Analyze data
    print("\n[Step 4/5] Analyzing data...")
    total_revenue = analyze_total_sales(df_clean)
    product_performance = analyze_product_performance(df_clean)
    regional_performance = analyze_regional_performance(df_clean)
    customer_insights = analyze_customer_insights(df_clean)
    analyze_price_quantity_relationship(df_clean)

    # Step 5: Generate report
    print("\n[Step 5/5] Generating report...")
    generate_report(df_clean)

    print("\n" + "=" * 60)
    print("✅ ANALYSIS COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    print("\n📊 Check 'analysis_report.md' for the full report")
    print("=" * 60 + "\n")

# Run the program
if __name__ == "__main__":
    main()


🎯 SALES DATA ANALYSIS PROGRAM

[Step 1/5] Loading data...
✅ Data loaded successfully!

[Step 2/5] Exploring data...

📊 BASIC DATASET INFORMATION

📏 Dataset Shape: 100 rows × 7 columns

📋 Column Information:
Date           object
Product        object
Quantity        int64
Price           int64
Customer_ID    object
Region         object
Total_Sales     int64
dtype: object

👀 First 5 Rows of Data:
         Date     Product  Quantity  Price Customer_ID Region  Total_Sales
0  2024-01-01       Phone         7  37300     CUST001   East       261100
1  2024-01-02  Headphones         4  15406     CUST002  North        61624
2  2024-01-03       Phone         2  21746     CUST003   West        43492
3  2024-01-04  Headphones         1  30895     CUST004   East        30895
4  2024-01-05      Laptop         8  39835     CUST005  North       318680

📈 Statistical Summary:
         Quantity         Price    Total_Sales
count  100.000000    100.000000     100.000000
mean     4.780000  25808.510000

In [3]:
with open('/content/analysis_report.md', 'r') as f:
    report_content = f.read()
print(report_content)

# Sales Data Analysis Report
Generated on: December 25, 2025

---

## Executive Summary

This report analyzes 100 sales transactions to provide insights into product performance, regional trends, and customer behavior.

### Key Metrics at a Glance
- **Total Revenue**: ₹12,365,048.00
- **Total Transactions**: 100
- **Unique Customers**: 100
- **Average Order Value**: ₹123,650.48

---

## 1. Product Performance

### Top Products by Revenue
| Product    |   Total_Quantity_Sold |   Total_Revenue |   Number_of_Orders |
|:-----------|----------------------:|----------------:|-------------------:|
| Laptop     |                   136 |         3889210 |                 24 |
| Tablet     |                   127 |         2884340 |                 26 |
| Phone      |                   101 |         2859394 |                 20 |
| Headphones |                    48 |         1384033 |                 15 |
| Monitor    |                    66 |         1348071 |                 15 |

### Key Fin